# 📑 Phân loại Fashion-MNIST - AI Project 3

Thành viên thực hiện: **Thành viên B**
Nhiệm vụ: Phân tích tập dữ liệu Fashion-MNIST, huấn luyện và so sánh mô hình **Decision Tree** và **Neural Network (MLP)**.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục cha vào sys.path để import helpers.py
sys.path.append(os.path.abspath('..'))
from utils.helpers import stratified_split, evaluate_model, plot_multiple_distributions

## 2.1 Chuẩn bị dữ liệu (Preparing the dataset)

Tải dữ liệu Fashion-MNIST từ OpenML và thực hiện phân chia tập huấn luyện (Train), kiểm định (Validation) theo tỷ lệ 80/20 phân tầng (stratified), giữ nguyên tập kiểm thử (Test) có sẵn.

In [ ]:
from sklearn.datasets import fetch_openml

print("Đang tải dữ liệu Fashion-MNIST từ OpenML (quá trình này có thể mất vài phút)...")
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X, y = fashion_mnist.data, fashion_mnist.target.astype(int)
print("Tải dữ liệu thành công!")
print(f"Kích thước X: {X.shape}, y: {y.shape}")

In [ ]:
# Chia tập Train_full (60,000 mẫu đầu) và tập Test (10,000 mẫu cuối)
X_train_full, y_train_full = X[:60000], y[:60000]
X_test, y_test = X[60000:], y[60000:]

# Chia tập Train_full thành Train (80%) và Validation (20%) bằng stratified split
X_train, y_train, X_val, y_val = stratified_split(X_train_full, y_train_full, val_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train.shape}, nhãn: {y_train.shape}")
print(f"Kích thước tập Validation: {X_val.shape}, nhãn: {y_val.shape}")
print(f"Kích thước tập Test: {X_test.shape}, nhãn: {y_test.shape}")

In [ ]:
# Khai báo tên các lớp (Class names)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
# Vẽ biểu đồ class distribution
os.makedirs('../figures', exist_ok=True)
plot_multiple_distributions(
    [y_train_full, y_train, y_val, y_test],
    ["Original Train", "Split Train", "Validation", "Test"],
    save_path="../figures/fashion_distribution.png"
)

## 2.2 Xây dựng cây quyết định (Building the decision tree classifier)

Fit mô hình `DecisionTreeClassifier` (sử dụng Entropy) trên tập Train và trực quan hóa cây bằng Graphviz.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import graphviz
from sklearn.tree import export_graphviz

print("Đang huấn luyện mô hình Decision Tree mặc định...")
dt_base = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_base.fit(X_train, y_train)
print(f"Độ chính xác trên Validation set (Mô hình mặc định): {dt_base.score(X_val, y_val):.4f}")

In [ ]:
print("Đang sinh sơ đồ cây quyết định (giới hạn độ sâu tối đa = 3 để dễ quan sát)...")
dot_data = export_graphviz(
    dt_base, 
    max_depth=3,
    feature_names=[f"pixel_{i}" for i in range(X_train.shape[1])],
    class_names=class_names,
    filled=True, 
    rounded=True,  
    special_characters=True
)
graph = graphviz.Source(dot_data)
graph.render("../figures/fashion_tree", format="png", cleanup=True)
print("Đã lưu sơ đồ cây quyết định vào figures/fashion_tree.png")
graph

## 2.3 Tinh chỉnh siêu tham số (Hyperparameter tuning for decision tree classifier)

Tìm cấu hình tốt nhất cho các siêu tham số `max_depth`, `min_samples_split`, và `min_samples_leaf` bằng cách đánh giá trên tập Validation.

In [ ]:
from sklearn.model_selection import ParameterGrid

# Thiết lập danh sách các siêu tham số cần tinh chỉnh
param_grid = {
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_val_acc = 0.0
best_params = {}
best_dt_model = None

print("Bắt đầu quá trình Tinh chỉnh Siêu tham số (Grid Search)...")
for params in ParameterGrid(param_grid):
    dt = DecisionTreeClassifier(
        criterion='entropy',
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        random_state=42
    )
    dt.fit(X_train, y_train)
    val_acc = dt.score(X_val, y_val)
    print(f"Params: {params} | Validation Accuracy: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params
        best_dt_model = dt

print("\n--- KẾT QUẢ TỐT NHẤT DỰA TRÊN TẬP VALIDATION ---")
print("Best Hyperparameters:", best_params)
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

## 2.4 Xây dựng bộ phân lớp mạng nơ-ron (Building the neural network classifier)

Huấn luyện mô hình `MLPClassifier` với ít nhất một ẩn lớp (hidden layer) và hàm kích hoạt phi tuyến, đánh giá hiệu năng trên tập Validation.

In [ ]:
from sklearn.neural_network import MLPClassifier

print("Đang huấn luyện mô hình Mạng Nơ-ron (MLP)...")
# Thiết lập mạng MLP: 1 hidden layer gồm 128 nơ-ron, hàm kích hoạt relu, solver adam
mlp = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42,
    verbose=True
)

mlp.fit(X_train, y_train)
mlp_val_acc = mlp.score(X_val, y_val)
print(f"Độ chính xác của MLP trên Validation set: {mlp_val_acc:.4f}")

## 2.5 Đánh giá và so sánh hiệu năng (Performance evaluation and comparison)

Đánh giá các mô hình tối ưu đã chọn trên tập dữ liệu kiểm thử độc lập (Test Set), tính toán `classification_report` và vẽ `confusion_matrix`.

In [ ]:
print("=== ĐÁNH GIÁ MÔ HÌNH DECISION TREE (TUNED) TRÊN TEST SET ===")
y_pred_dt, dt_test_acc = evaluate_model(
    best_dt_model, 
    X_test, 
    y_test, 
    model_name="Decision Tree (Tuned)", 
    save_cm_path="../figures/fashion_dt_cm.png",
    class_names=class_names
)

In [ ]:
print("=== ĐÁNH GIÁ MÔ HÌNH NEURAL NETWORK (MLP) TRÊN TEST SET ===")
y_pred_mlp, mlp_test_acc = evaluate_model(
    mlp, 
    X_test, 
    y_test, 
    model_name="Neural Network (MLP)", 
    save_cm_path="../figures/fashion_mlp_cm.png",
    class_names=class_names
)

## Nhận xét và Kết luận (Insights & Conclusion)

1. **Độ chính xác (Accuracy):** Mạng nơ-ron MLP mang lại kết quả tốt hơn rõ rệt so với Decision Tree trên tập dữ liệu hình ảnh thời trang Fashion-MNIST. Điều này phản ánh khả năng vượt trội của MLP trong việc tự động trích xuất các đặc trưng phân tầng phức tạp.
2. **Ma trận nhầm lẫn (Confusion Matrix):** Các lỗi chủ yếu tập trung vào các lớp trang phục có hình dáng tương tự nhau như **Shirt** (hay bị nhầm với T-shirt/top, Coat và Pullover). Việc phân biệt các chi tiết như khuy áo, cổ áo đòi hỏi khả năng học phi tuyến sâu sắc mà Decision Tree khó đáp ứng được khi bị khống chế độ sâu cây.
3. **Độ phức tạp:** Decision Tree học nhanh và trực quan trực tiếp cấu trúc cây ra được, tuy nhiên dễ bị quá khớp (overfit) và có hiệu suất hạn chế trên dữ liệu ảnh số lớn. MLP cho hiệu năng cao hơn nhưng tốn nhiều thời gian huấn luyện và cấu trúc mô hình có độ minh bạch thấp (Black-box).